# 02 – Data Cleaning and Preprocessing

## 1. Introduction

Data cleaning is an important step in the data analysis process. Raw customer review data may contain missing values, unnecessary attributes, inconsistent text formats, and other data quality issues.

In this notebook, the Consumer Reviews dataset will be cleaned and prepared for further analysis. The main focus is to improve data quality while preserving useful customer feedback information.

## 2. Objectives

The objectives of this notebook are:

1. Load the original Consumer Reviews dataset.
2. Examine missing values and data quality.
3. Identify and handle duplicate records.
4. Remove unnecessary columns.
5. Clean and standardize customer review text.
6. Validate ratings and review dates.
7. Save the cleaned dataset for further analysis.

## 3. Dataset Information

- Dataset Name: Datafiniti Amazon Product Consumer Reviews
- Original Records: 28,332
- Original Attributes: 24
- Data Format: CSV

## 4. Expected Outcome

A cleaned and structured customer reviews dataset suitable for sentiment analysis, theme identification, and further customer intelligence analysis.

In [1]:
# Import required libraries

import pandas as pd
import numpy as np
import re
from pathlib import Path

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Detect project root directory

current_path = Path.cwd()
PROJECT_ROOT = None

for path in [current_path, *current_path.parents]:
    if (path / "data").is_dir() and (path / "notebooks").is_dir():
        PROJECT_ROOT = path
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Project folder not found.")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

DATASET_PATH = RAW_DIR / "Consumer_Reviews.csv"

print("Project Root:", PROJECT_ROOT)
print("Raw Data Folder:", RAW_DIR)
print("Processed Data Folder:", PROCESSED_DIR)
print("Dataset Exists:", DATASET_PATH.exists())

Project Root: C:\Users\anant\OneDrive\Desktop\AI_Customer_Intelligence
Raw Data Folder: C:\Users\anant\OneDrive\Desktop\AI_Customer_Intelligence\data\raw
Processed Data Folder: C:\Users\anant\OneDrive\Desktop\AI_Customer_Intelligence\data\processed
Dataset Exists: True


## 5. Loading the Original Dataset

The original Consumer Reviews dataset is loaded from the raw data directory. The raw dataset is kept unchanged to maintain data integrity and allow comparison between the original and cleaned data.

The dataset will be loaded into a Pandas DataFrame for further preprocessing.

In [3]:
# Load original dataset

df = pd.read_csv(DATASET_PATH, low_memory=False)

print("Dataset loaded successfully!")
print("--------------------------------")
print("Number of Records:", df.shape[0])
print("Number of Attributes:", df.shape[1])

display(df.head())

Dataset loaded successfully!
--------------------------------
Number of Records: 28332
Number of Attributes: 24


,id,dateAdded,dateUpdated,name,asins,brand,categories,primaryCategories,imageURLs,keys,...,reviews.didPurchase,reviews.doRecommend,reviews.id,reviews.numHelpful,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.username,sourceURLs
0,AVpgNzjwLJeJML43Kpxn,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...",Health & Beauty,https://images-na.ssl-images-amazon.com/images...,"amazonbasics/hl002619,amazonbasicsaaaperforman...",...,NaN,NaN,NaN,NaN,3,https://www.amazon.com/product-reviews/B00QWO9...,I order 3 of them and one of the item is bad q...,... 3 of them and one of the item is bad quali...,Byger yang,"https://www.barcodable.com/upc/841710106442,ht..."
1,AVpgNzjwLJeJML43Kpxn,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...",Health & Beauty,https://images-na.ssl-images-amazon.com/images...,"amazonbasics/hl002619,amazonbasicsaaaperforman...",...,NaN,NaN,NaN,NaN,4,https://www.amazon.com/product-reviews/B00QWO9...,Bulk is always the less expensive way to go fo...,... always the less expensive way to go for pr...,ByMG,"https://www.barcodable.com/upc/841710106442,ht..."
2,AVpgNzjwLJeJML43Kpxn,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...",Health & Beauty,https://images-na.ssl-images-amazon.com/images...,"amazonbasics/hl002619,amazonbasicsaaaperforman...",...,NaN,NaN,NaN,NaN,5,https://www.amazon.com/product-reviews/B00QWO9...,Well they are not Duracell but for the price i...,... are not Duracell but for the price i am ha...,BySharon Lambert,"https://www.barcodable.com/upc/841710106442,ht..."
3,AVpgNzjwLJeJML43Kpxn,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...",Health & Beauty,https://images-na.ssl-images-amazon.com/images...,"amazonbasics/hl002619,amazonbasicsaaaperforman...",...,NaN,NaN,NaN,NaN,5,https://www.amazon.com/product-reviews/B00QWO9...,Seem to work as well as name brand batteries a...,... as well as name brand batteries at a much ...,Bymark sexson,"https://www.barcodable.com/upc/841710106442,ht..."
4,AVpgNzjwLJeJML43Kpxn,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...",Health & Beauty,https://images-na.ssl-images-amazon.com/images...,"amazonbasics/hl002619,amazonbasicsaaaperforman...",...,NaN,NaN,NaN,NaN,5,https://www.amazon.com/product-reviews/B00QWO9...,These batteries are very long lasting the pric...,... batteries are very long lasting the price ...,Bylinda,"https://www.barcodable.com/upc/841710106442,ht..."


## 6. Understanding the Dataset Structure

The structure of the dataset is examined to understand its attributes, data types, and non-null values. This step helps identify columns that may require preprocessing or special handling during the cleaning process.

In [4]:
# Examine dataset information

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 28332 entries, 0 to 28331
Data columns (total 24 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   28332 non-null  str    
 1   dateAdded            28332 non-null  str    
 2   dateUpdated          28332 non-null  str    
 3   name                 28332 non-null  str    
 4   asins                28332 non-null  str    
 5   brand                28332 non-null  str    
 6   categories           28332 non-null  str    
 7   primaryCategories    28332 non-null  str    
 8   imageURLs            28332 non-null  str    
 9   keys                 28332 non-null  str    
 10  manufacturer         28332 non-null  str    
 11  manufacturerNumber   28332 non-null  str    
 12  reviews.date         28332 non-null  str    
 13  reviews.dateSeen     28332 non-null  str    
 14  reviews.didPurchase  9 non-null      object 
 15  reviews.doRecommend  16086 non-null  object 
 1

## 7. Missing Values Analysis

Missing values are examined to identify incomplete information in the dataset. The number and percentage of missing values in each attribute are calculated to understand data quality.

This analysis helps determine suitable preprocessing strategies, such as retaining, removing, or handling incomplete attributes.

In [6]:
# Analyze missing values

missing_analysis = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing Percentage": (
        df.isnull().sum() / len(df)
    ) * 100
})

missing_analysis = missing_analysis.sort_values(
    by="Missing Percentage",
    ascending=False
)

display(missing_analysis.round(2))

,Missing Values,Missing Percentage
reviews.didPurchase,28323,99.97
reviews.id,28291,99.86
reviews.doRecommend,12246,43.22
reviews.numHelpful,12217,43.12
reviews.username,5,0.02
id,0,0.00
dateUpdated,0,0.00
dateAdded,0,0.00
primaryCategories,0,0.00
categories,0,0.00


In [7]:
# Display only columns containing missing values

missing_columns = missing_analysis[
    missing_analysis["Missing Values"] > 0
]

print("Total Columns with Missing Values:",
      len(missing_columns))

display(missing_columns.round(2))

Total Columns with Missing Values: 5


,Missing Values,Missing Percentage
reviews.didPurchase,28323,99.97
reviews.id,28291,99.86
reviews.doRecommend,12246,43.22
reviews.numHelpful,12217,43.12
reviews.username,5,0.02


## 8. Missing Values: Findings and Treatment Strategy

The missing value analysis identified five attributes containing incomplete records.

Based on the percentage of missing values and their relevance to customer feedback analysis, the following treatment strategy was selected:

1. The `reviews.didPurchase` attribute will be removed because 99.97% of its values are missing.
2. The `reviews.id` attribute will be removed because 99.86% of its values are missing.
3. The `reviews.doRecommend` attribute will be retained because recommendation information may support customer preference analysis.
4. The `reviews.numHelpful` attribute will be retained because helpful vote counts may provide additional insights.
5. The five missing values in `reviews.username` will be replaced with the label "Unknown".

Missing values in recommendation and helpful vote attributes will remain unchanged because missing information should not automatically be interpreted as False or zero.

The original dataset will remain unchanged. All preprocessing operations will be performed on a separate copy.

In [9]:
# Create a separate working copy

clean_df = df.copy()

# Remove attributes with extremely high missing values

columns_to_remove = [
    "reviews.didPurchase",
    "reviews.id"
]

clean_df.drop(
    columns=columns_to_remove,
    inplace=True
)

# Handle missing usernames

clean_df["reviews.username"] = (
    clean_df["reviews.username"].fillna("Unknown")
)

print("Missing value treatment completed!")
print("-----------------------------------")
print("Original Dataset Shape:", df.shape)
print("Working Dataset Shape:", clean_df.shape)

print("\nRemaining Missing Values:")
display(
    clean_df.isnull().sum()[
        clean_df.isnull().sum() > 0
    ].to_frame(name="Missing Values")
)

Missing value treatment completed!
-----------------------------------
Original Dataset Shape: (28332, 24)
Working Dataset Shape: (28332, 22)

Remaining Missing Values:


,Missing Values
reviews.doRecommend,12246
reviews.numHelpful,12217


In [10]:
# Verify missing value treatment

print("MISSING VALUE TREATMENT VERIFICATION")
print("------------------------------------")

print("Missing didPurchase column:",
      "reviews.didPurchase" in clean_df.columns)

print("Missing reviews.id column:",
      "reviews.id" in clean_df.columns)

print("Missing usernames:",
      clean_df["reviews.username"].isnull().sum())

print("Remaining missing values:",
      clean_df.isnull().sum().sum())

MISSING VALUE TREATMENT VERIFICATION
------------------------------------
Missing didPurchase column: False
Missing reviews.id column: False
Missing usernames: 0
Remaining missing values: 24463


## 9. Duplicate Records Analysis

Duplicate records can affect the accuracy of data analysis by causing repeated information to be counted multiple times.

In this step, the working dataset is examined to identify duplicate rows. The number and percentage of duplicate records are calculated to evaluate the uniqueness of the dataset.

In [11]:
# Identify duplicate records

total_records = len(clean_df)

duplicate_count = clean_df.duplicated().sum()

duplicate_percentage = (
    duplicate_count / total_records
) * 100

print("DUPLICATE RECORD ANALYSIS")
print("-------------------------")
print("Total Records:", total_records)
print("Duplicate Records:", duplicate_count)
print("Duplicate Percentage:",
      round(duplicate_percentage, 2), "%")

DUPLICATE RECORD ANALYSIS
-------------------------
Total Records: 28332
Duplicate Records: 0
Duplicate Percentage: 0.0 %


In [12]:
# Display duplicate records for inspection

duplicate_records = clean_df[
    clean_df.duplicated(keep=False)
]

print("Total Duplicate Rows for Inspection:",
      len(duplicate_records))

display(duplicate_records.head(10))

Total Duplicate Rows for Inspection: 0


,id,dateAdded,dateUpdated,name,asins,brand,categories,primaryCategories,imageURLs,keys,...,reviews.date,reviews.dateSeen,reviews.doRecommend,reviews.numHelpful,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.username,sourceURLs


## 10. Duplicate Records: Findings and Conclusion

The duplicate record analysis was performed on the working dataset using Pandas.

The analysis identified zero duplicate records among the 28,332 customer reviews.

Since no duplicate rows were found, no records were removed during this stage.

All 28,332 records have been retained for further preprocessing and analysis.

## 11. Review Text Quality Analysis

Customer review text is the primary input for Natural Language Processing (NLP) tasks in this project.

Before performing text preprocessing, the review content is examined to identify empty strings, whitespace-only values, and other text quality issues.

This analysis helps ensure that customer feedback is suitable for sentiment analysis and theme identification.

In [13]:
# Analyze review text quality

print("REVIEW TEXT QUALITY ANALYSIS")
print("----------------------------")

print("Total Reviews:",
      len(clean_df))

print("Missing Review Text:",
      clean_df["reviews.text"].isnull().sum())

empty_reviews = (
    clean_df["reviews.text"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("Empty or Whitespace-only Reviews:",
      empty_reviews)

print("Unique Review Texts:",
      clean_df["reviews.text"].nunique())

REVIEW TEXT QUALITY ANALYSIS
----------------------------
Total Reviews: 28332
Missing Review Text: 0
Empty or Whitespace-only Reviews: 0
Unique Review Texts: 18168


In [14]:
# Inspect sample customer reviews

display(
    clean_df[["reviews.title", "reviews.text", "reviews.rating"]]
    .head(10)
)

,reviews.title,reviews.text,reviews.rating
0,... 3 of them and one of the item is bad quali...,I order 3 of them and one of the item is bad q...,3
1,... always the less expensive way to go for pr...,Bulk is always the less expensive way to go fo...,4
2,... are not Duracell but for the price i am ha...,Well they are not Duracell but for the price i...,5
3,... as well as name brand batteries at a much ...,Seem to work as well as name brand batteries a...,5
4,... batteries are very long lasting the price ...,These batteries are very long lasting the pric...,5
5,... batteries for Christmas and the AmazonBasi...,Bought a lot of batteries for Christmas and th...,5
6,... batteries have ordered them in the past be...,ive not had any problame with these batteries ...,5
7,... batteries that last quite a while then the...,Well if you are looking for cheap non-recharge...,5
8,... do not hold the amount of high power juice...,These do not hold the amount of high power jui...,3
9,... done well by me appear to have a good shel...,AmazonBasics AA AAA batteries have done well b...,4


## 12. Text Cleaning and Preprocessing

Customer reviews may contain inconsistent text formatting, HTML tags, URLs, and unnecessary whitespace.

To prepare the review data for Natural Language Processing, a separate cleaned text column is created.

The preprocessing operations include:

1. Converting text into lowercase.
2. Removing HTML tags.
3. Removing URLs.
4. Removing unnecessary whitespace.
5. Preserving punctuation to retain useful sentiment-related information.

The original review text is preserved to maintain data integrity and allow comparison with the preprocessed text.

In [15]:
# Function to clean customer review text

def clean_review_text(text):

    text = str(text)

    # Convert text to lowercase
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Apply text cleaning

clean_df["review_text_clean"] = (
    clean_df["reviews.text"].apply(clean_review_text)
)

print("Text preprocessing completed successfully!")
print("-----------------------------------------")
print("Total Reviews:", len(clean_df))
print("Original Text Column: reviews.text")
print("Cleaned Text Column: review_text_clean")

display(
    clean_df[
        ["reviews.text", "review_text_clean"]
    ].head(10)
)

Text preprocessing completed successfully!
-----------------------------------------
Total Reviews: 28332
Original Text Column: reviews.text
Cleaned Text Column: review_text_clean


,reviews.text,review_text_clean
0,I order 3 of them and one of the item is bad q...,i order 3 of them and one of the item is bad q...
1,Bulk is always the less expensive way to go fo...,bulk is always the less expensive way to go fo...
2,Well they are not Duracell but for the price i...,well they are not duracell but for the price i...
3,Seem to work as well as name brand batteries a...,seem to work as well as name brand batteries a...
4,These batteries are very long lasting the pric...,these batteries are very long lasting the pric...
5,Bought a lot of batteries for Christmas and th...,bought a lot of batteries for christmas and th...
6,ive not had any problame with these batteries ...,ive not had any problame with these batteries ...
7,Well if you are looking for cheap non-recharge...,well if you are looking for cheap non-recharge...
8,These do not hold the amount of high power jui...,these do not hold the amount of high power jui...
9,AmazonBasics AA AAA batteries have done well b...,amazonbasics aa aaa batteries have done well b...


## 13. Verification of Cleaned Review Text

After applying text preprocessing operations, the cleaned review text is validated to ensure that the transformation was successful.

The verification checks include:

1. Checking missing values in the cleaned text column.
2. Checking empty or whitespace-only reviews.
3. Confirming that all cleaned text entries are in lowercase.
4. Comparing the original and cleaned review text.

These checks help ensure that the preprocessing process has maintained the quality and completeness of customer feedback.

In [16]:
# Verify cleaned review text

missing_clean_text = clean_df["review_text_clean"].isnull().sum()

empty_clean_text = (
    clean_df["review_text_clean"]
    .str.strip()
    .eq("")
    .sum()
)

lowercase_check = (
    clean_df["review_text_clean"]
    .eq(clean_df["review_text_clean"].str.lower())
    .all()
)

print("CLEANED TEXT VERIFICATION")
print("-------------------------")
print("Total Reviews:", len(clean_df))
print("Missing Cleaned Text:", missing_clean_text)
print("Empty Cleaned Text:", empty_clean_text)
print("All Text in Lowercase:", lowercase_check)
print("Original Text Preserved:",
      "reviews.text" in clean_df.columns)

CLEANED TEXT VERIFICATION
-------------------------
Total Reviews: 28332
Missing Cleaned Text: 0
Empty Cleaned Text: 0
All Text in Lowercase: True
Original Text Preserved: True


## 14. Rating Validation

Customer ratings are an important indicator of customer satisfaction.

In this step, the rating column is examined to identify invalid or unexpected values. The dataset is expected to contain ratings ranging from 1 to 5.

The validation process checks the minimum rating, maximum rating, missing values, and frequency distribution of each rating.

In [17]:
# Validate customer ratings

print("RATING VALIDATION")
print("-----------------")

print("Total Records:", len(clean_df))
print("Missing Ratings:",
      clean_df["reviews.rating"].isnull().sum())

print("Minimum Rating:",
      clean_df["reviews.rating"].min())

print("Maximum Rating:",
      clean_df["reviews.rating"].max())

print("\nUnique Rating Values:")
print(sorted(clean_df["reviews.rating"].unique()))

print("\nRating Distribution:")
display(
    clean_df["reviews.rating"]
    .value_counts()
    .sort_index()
    .to_frame(name="Number of Reviews")
)

RATING VALIDATION
-----------------
Total Records: 28332
Missing Ratings: 0
Minimum Rating: 1
Maximum Rating: 5

Unique Rating Values:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Rating Distribution:


,Number of Reviews
reviews.rating,
1,965
2,616
3,1206
4,5648
5,19897


## 15. Rating Validation: Findings and Conclusion

The rating validation process confirmed that all 28,332 records contain valid customer ratings.

The minimum rating is 1 and the maximum rating is 5. No missing or invalid rating values were identified.

Since all ratings fall within the expected range, no rating values were modified or removed.

The rating attribute is suitable for customer satisfaction analysis and rating-based comparisons.

In [18]:
# Identify ratings outside the valid range

invalid_ratings = clean_df[
    ~clean_df["reviews.rating"].between(1, 5)
]

print("INVALID RATING VERIFICATION")
print("---------------------------")
print("Invalid Rating Records:", len(invalid_ratings))

display(
    invalid_ratings[["reviews.rating", "reviews.text"]].head()
)

INVALID RATING VERIFICATION
---------------------------
Invalid Rating Records: 0


,reviews.rating,reviews.text


## 16. Review Date Validation

Review dates are important for understanding customer feedback trends over time.

The review date attribute is converted into a standardized datetime format using mixed-format parsing. This approach supports different date representations within the same column.

The converted dates are checked for missing or invalid values to ensure their suitability for time-based analysis.

In [20]:
# Convert review dates into datetime format

clean_df["review_date"] = pd.to_datetime(
    clean_df["reviews.date"],
    format="mixed",
    errors="coerce",
    utc=True
)

valid_dates = clean_df["review_date"].notna().sum()
invalid_dates = clean_df["review_date"].isna().sum()

print("REVIEW DATE VALIDATION")
print("----------------------")
print("Total Records:", len(clean_df))
print("Valid Dates:", valid_dates)
print("Invalid Dates:", invalid_dates)
print(
    "Date Conversion Success:",
    round((valid_dates / len(clean_df)) * 100, 2),
    "%"
)

REVIEW DATE VALIDATION
----------------------
Total Records: 28332
Valid Dates: 28332
Invalid Dates: 0
Date Conversion Success: 100.0 %


## 17. Final Data Quality Verification

After completing the data cleaning and preprocessing operations, a final quality assessment is performed on the working dataset.

This assessment verifies the total number of records, attributes, duplicate rows, missing values, review text quality, rating validity, and date conversion results.

The purpose is to ensure that the processed dataset is consistent and ready for subsequent sentiment analysis and theme identification.

In [21]:
# Final data quality verification

print("FINAL DATA QUALITY REPORT")
print("=" * 40)

print("Total Records:", len(clean_df))
print("Total Attributes:", clean_df.shape[1])

print("Duplicate Records:", clean_df.duplicated().sum())

print("Missing Review Text:",
      clean_df["reviews.text"].isnull().sum())

print("Missing Cleaned Text:",
      clean_df["review_text_clean"].isnull().sum())

print("Missing Ratings:",
      clean_df["reviews.rating"].isnull().sum())

print("Invalid Ratings:",
      (~clean_df["reviews.rating"].between(1, 5)).sum())

print("Valid Review Dates:",
      clean_df["review_date"].notna().sum())

print("Invalid Review Dates:",
      clean_df["review_date"].isnull().sum())

print("Remaining Missing Values:",
      clean_df.isnull().sum().sum())

FINAL DATA QUALITY REPORT
Total Records: 28332
Total Attributes: 24
Duplicate Records: 0
Missing Review Text: 0
Missing Cleaned Text: 0
Missing Ratings: 0
Invalid Ratings: 0
Valid Review Dates: 28332
Invalid Review Dates: 0
Remaining Missing Values: 24463


## 18. Exporting the Cleaned Dataset

The cleaned dataset is exported to the processed data directory in CSV format.

The original dataset remains unchanged, while the processed dataset contains the cleaned review text, standardized review dates, and selected attributes required for further analysis.

This exported file will serve as the input dataset for sentiment analysis and theme identification.

In [22]:
# Save the cleaned dataset

CLEANED_DATASET_PATH = (
    PROCESSED_DIR / "cleaned_customer_reviews.csv"
)

clean_df.to_csv(
    CLEANED_DATASET_PATH,
    index=False
)

print("Cleaned dataset saved successfully!")
print("File Location:", CLEANED_DATASET_PATH)
print("Final Dataset Shape:", clean_df.shape)

Cleaned dataset saved successfully!
File Location: C:\Users\anant\OneDrive\Desktop\AI_Customer_Intelligence\data\processed\cleaned_customer_reviews.csv
Final Dataset Shape: (28332, 24)


## 19. Verification of Exported Dataset

The exported CSV file is reloaded to verify that the cleaned dataset has been saved correctly.

The number of records, attributes, and important preprocessing columns are checked to confirm that the exported file is ready for subsequent analysis.

In [23]:
# Verify exported dataset

verified_df = pd.read_csv(
    CLEANED_DATASET_PATH,
    low_memory=False
)

print("EXPORTED DATASET VERIFICATION")
print("-----------------------------")

print("File Exists:", CLEANED_DATASET_PATH.exists())
print("Verified Dataset Shape:", verified_df.shape)

print("Cleaned Text Column Exists:",
      "review_text_clean" in verified_df.columns)

print("Review Date Column Exists:",
      "review_date" in verified_df.columns)

print("\nFirst Five Records:")
display(verified_df.head())

EXPORTED DATASET VERIFICATION
-----------------------------
File Exists: True
Verified Dataset Shape: (28332, 24)
Cleaned Text Column Exists: True
Review Date Column Exists: True

First Five Records:


,id,dateAdded,dateUpdated,name,asins,brand,categories,primaryCategories,imageURLs,keys,...,reviews.doRecommend,reviews.numHelpful,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.username,sourceURLs,review_text_clean,review_date
0,AVpgNzjwLJeJML43Kpxn,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...",Health & Beauty,https://images-na.ssl-images-amazon.com/images...,"amazonbasics/hl002619,amazonbasicsaaaperforman...",...,NaN,NaN,3,https://www.amazon.com/product-reviews/B00QWO9...,I order 3 of them and one of the item is bad q...,... 3 of them and one of the item is bad quali...,Byger yang,"https://www.barcodable.com/upc/841710106442,ht...",i order 3 of them and one of the item is bad q...,2017-03-02 00:00:00+00:00
1,AVpgNzjwLJeJML43Kpxn,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...",Health & Beauty,https://images-na.ssl-images-amazon.com/images...,"amazonbasics/hl002619,amazonbasicsaaaperforman...",...,NaN,NaN,4,https://www.amazon.com/product-reviews/B00QWO9...,Bulk is always the less expensive way to go fo...,... always the less expensive way to go for pr...,ByMG,"https://www.barcodable.com/upc/841710106442,ht...",bulk is always the less expensive way to go fo...,2016-08-31 00:00:00+00:00
2,AVpgNzjwLJeJML43Kpxn,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...",Health & Beauty,https://images-na.ssl-images-amazon.com/images...,"amazonbasics/hl002619,amazonbasicsaaaperforman...",...,NaN,NaN,5,https://www.amazon.com/product-reviews/B00QWO9...,Well they are not Duracell but for the price i...,... are not Duracell but for the price i am ha...,BySharon Lambert,"https://www.barcodable.com/upc/841710106442,ht...",well they are not duracell but for the price i...,2016-12-06 00:00:00+00:00
3,AVpgNzjwLJeJML43Kpxn,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...",Health & Beauty,https://images-na.ssl-images-amazon.com/images...,"amazonbasics/hl002619,amazonbasicsaaaperforman...",...,NaN,NaN,5,https://www.amazon.com/product-reviews/B00QWO9...,Seem to work as well as name brand batteries a...,... as well as name brand batteries at a much ...,Bymark sexson,"https://www.barcodable.com/upc/841710106442,ht...",seem to work as well as name brand batteries a...,2016-03-28 00:00:00+00:00
4,AVpgNzjwLJeJML43Kpxn,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...",Health & Beauty,https://images-na.ssl-images-amazon.com/images...,"amazonbasics/hl002619,amazonbasicsaaaperforman...",...,NaN,NaN,5,https://www.amazon.com/product-reviews/B00QWO9...,These batteries are very long lasting the pric...,... batteries are very long lasting the price ...,Bylinda,"https://www.barcodable.com/upc/841710106442,ht...",these batteries are very long lasting the pric...,2017-04-22 00:00:00+00:00
